# ModernFloraBERT regression fine-tuning (Colab)

Run this notebook from VS Code with a Google Colab Python kernel after the plant-to-maize MLM notebook has produced its checkpoint. It supports both sides of the clean ablation through `FLORABERT_REGRESSION_SOURCE`: `plant` runs the control and `maize` consumes the continued-MLM checkpoint.

The existing mean-pooling regression API, natural-log target `ln(TPM + 0.001)`, validation selection, and fixed metric semantics are preserved. The plant and maize regression outputs are always written to separate directories.

Run this notebook once with `FLORABERT_REGRESSION_SOURCE=plant` and once with `FLORABERT_REGRESSION_SOURCE=maize` when comparing the ablation.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


drive_root = os.environ.get('FLORABERT_DRIVE_ROOT', '').strip()
if drive_root:
    if drive_root.startswith('/content/drive'):
        try:
            from google.colab import drive

            if not Path('/content/drive/MyDrive').exists():
                drive.mount('/content/drive')
        except ImportError as exc:
            raise RuntimeError(
                'FLORABERT_DRIVE_ROOT was set, but this is not a Colab runtime'
            ) from exc
    run_root = Path(drive_root).expanduser()
else:
    run_root = Path(
        os.environ.get('FLORABERT_RUN_ROOT', '/content/florabert_runs')
    ).expanduser()

repo_dir = Path(
    os.environ.get('FLORABERT_REPO_DIR', '/content/florabert')
).expanduser()
repo_url = os.environ.get(
    'FLORABERT_REPO_URL',
    'https://github.com/gurveersinghvirk/florabert.git',
)
repo_ref = os.environ.get(
    'FLORABERT_REPO_REF',
    'feat/modernbert-maize-mlm-ablation',
)
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '').strip()

if not (repo_dir / '.git').is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f'{repo_dir} exists but is not an empty git checkout'
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--branch', repo_ref, '--depth', '1', repo_url, str(repo_dir)],
        check=True,
    )

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repo_dir,
    text=True,
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; expected {expected_commit}'
    )

run_root.mkdir(parents=True, exist_ok=True)
data_root = run_root / 'data'
genex_dir = data_root / 'genex' / 'nam'
kaggle_root = run_root / 'kaggle-modernflorabert-base-v3'
plant_checkpoint = kaggle_root / 'plant-checkpoint-6000'
plant_tokenizer = kaggle_root / 'modernbert-tokenizer'
maize_lm_checkpoint = Path(
    os.environ.get(
        'FLORABERT_MAIZE_LM_CHECKPOINT',
        str(run_root / 'models' / 'transformer' / 'language-model-modernbert-maize'),
    )
).expanduser()
plant_regression_output = run_root / 'prediction-model-modernbert-plant'
maize_regression_output = run_root / 'prediction-model-modernbert-maize'

for path in [
    data_root,
    genex_dir,
    kaggle_root,
    plant_checkpoint,
    plant_tokenizer,
    plant_regression_output,
    maize_regression_output,
]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))

print('Repo:', repo_dir)
print('Repo ref:', repo_ref)
print('Repo commit:', actual_commit)
print('Run root:', run_root)
print('Regression data directory:', genex_dir)
print('Plant checkpoint directory:', plant_checkpoint)
print('Plant tokenizer directory:', plant_tokenizer)
print('Maize MLM checkpoint directory:', maize_lm_checkpoint)
print('Plant regression output:', plant_regression_output)
print('Maize regression output:', maize_regression_output)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'requirements.txt'),
    ],
    cwd=repo_dir,
)

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'kagglehub>=1.0',
        'wandb>=0.19',
    ]
)

import importlib.metadata as importlib_metadata
import torch

for package_name in [
    'torch',
    'transformers',
    'datasets',
    'accelerate',
    'huggingface-hub',
    'kagglehub',
    'wandb',
]:
    try:
        print(package_name, importlib_metadata.version(package_name))
    except importlib_metadata.PackageNotFoundError:
        print(package_name, 'not found')

cuda_count = torch.cuda.device_count()
print('CUDA device count:', cuda_count)
if cuda_count:
    for device_idx in range(cuda_count):
        print(f'CUDA {device_idx}: {torch.cuda.get_device_name(device_idx)}')
else:
    print('No CUDA device is visible; training will use CPU and may be slow.')

In [ ]:
from huggingface_hub import login as hf_login


def runtime_secret(name):
    value = os.environ.get(name)
    if value and value.strip():
        return value.strip()

    try:
        from google.colab import userdata

        value = userdata.get(name)
    except Exception:
        value = None

    return value.strip() if value and value.strip() else None


hf_token = runtime_secret('HF_TOKEN') or runtime_secret('HUGGINGFACE_TOKEN')
if hf_token:
    hf_login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authentication loaded from a runtime secret.')
else:
    print('No Hugging Face token supplied; regression artifacts are public.')


kaggle_token = runtime_secret('KAGGLE_API_TOKEN')
if kaggle_token:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
else:
    cached_kaggle_files = [
        Path.home() / '.kaggle' / 'access_token',
        Path.home() / '.kaggle' / 'kaggle.json',
    ]
    if not any(path.is_file() for path in cached_kaggle_files):
        from getpass import getpass

        entered_kaggle_token = getpass(
            'Kaggle API token (hidden input; not saved in this notebook): '
        )
        if not entered_kaggle_token.strip():
            raise RuntimeError('No Kaggle API token was entered.')

        os.environ['KAGGLE_API_TOKEN'] = entered_kaggle_token.strip()
        del entered_kaggle_token

import kagglehub
print('KaggleHub is ready for the public version-3 dataset.')

In [ ]:
import wandb


wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    from getpass import getpass

    wandb_key = getpass(
        'Weights & Biases API key (hidden input; not saved in this notebook): '
    ).strip()

if not wandb_key:
    raise RuntimeError('No W&B API key was entered.')

os.environ['WANDB_API_KEY'] = wandb_key
os.environ.setdefault(
    'WANDB_PROJECT',
    'florabert-modernbert-maize-ablation',
)
wandb.login(relogin=True)
del wandb_key

print('W&B authentication is ready.')
print('W&B project:', os.environ['WANDB_PROJECT'])

In [ ]:
# Download the plant ModernBERT checkpoint, its unchanged tokenizer, and
# the existing NAM train/eval/test files from Kaggle dataset version 3.
kaggle_dataset = 'gurveersinghvirk/modernflorabert-base/versions/3'
kagglehub_stage_dir = kaggle_root / '.kagglehub-files'
kagglehub_stage_dir.mkdir(parents=True, exist_ok=True)
force_kaggle_download = env_bool('FLORABERT_FORCE_KAGGLE_DOWNLOAD', False)

kaggle_specs = [
    (
        'florabert/models/transformer/language-model-modernbert/config.json',
        plant_checkpoint,
    ),
    (
        'florabert/models/transformer/language-model-modernbert/model.safetensors',
        plant_checkpoint,
    ),
    (
        'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer.json',
        plant_tokenizer,
    ),
    (
        'florabert/models/modernbert-byte-level-bpe-tokenizer/tokenizer_config.json',
        plant_tokenizer,
    ),
    (
        'florabert/models/modernbert-byte-level-bpe-tokenizer/special_tokens_map.json',
        plant_tokenizer,
    ),
    (
        'florabert/data/final/transformer/genex/nam/train.tsv',
        genex_dir,
    ),
    (
        'florabert/data/final/transformer/genex/nam/eval.tsv',
        genex_dir,
    ),
    (
        'florabert/data/final/transformer/genex/nam/test.tsv',
        genex_dir,
    ),
]


def download_kaggle_file(remote_name, destination_dir):
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    local_path = destination_dir / Path(remote_name).name

    if (
        local_path.is_file()
        and local_path.stat().st_size > 0
        and not force_kaggle_download
    ):
        print('Reusing', local_path, 'bytes=', local_path.stat().st_size)
        return local_path

    downloaded_path = Path(
        kagglehub.dataset_download(
            kaggle_dataset,
            path=remote_name,
            output_dir=str(kagglehub_stage_dir),
            force_download=force_kaggle_download,
        )
    )

    candidates = [downloaded_path, kagglehub_stage_dir / remote_name]
    if downloaded_path.is_dir():
        candidates.extend(downloaded_path.rglob(Path(remote_name).name))

    source_path = next((path for path in candidates if path.is_file()), None)
    if source_path is None or source_path.stat().st_size == 0:
        raise FileNotFoundError(
            f'kagglehub did not produce a non-empty file for {remote_name}; '
            f'returned {downloaded_path}'
        )

    if source_path.resolve() != local_path.resolve():
        shutil.copy2(source_path, local_path)

    if not local_path.is_file() or local_path.stat().st_size == 0:
        raise IOError(f'Incomplete KaggleHub download: {local_path}')

    print(remote_name, '->', local_path, 'bytes=', local_path.stat().st_size)
    return local_path


print('Kaggle dataset:', kaggle_dataset)
for remote_name, destination_dir in kaggle_specs:
    download_kaggle_file(remote_name, destination_dir)

print('Resolved plant checkpoint:', plant_checkpoint)
print('Resolved plant tokenizer:', plant_tokenizer)
print('Resolved regression data:', genex_dir)

In [ ]:
regression_source = os.environ.get(
    'FLORABERT_REGRESSION_SOURCE',
    'maize',
).strip().lower()
if regression_source not in {'plant', 'maize'}:
    raise ValueError(
        'FLORABERT_REGRESSION_SOURCE must be plant or maize, got '
        f'{regression_source!r}'
    )

if regression_source == 'plant':
    source_checkpoint = plant_checkpoint
    regression_output = plant_regression_output
else:
    source_checkpoint = maize_lm_checkpoint
    regression_output = maize_regression_output

if not (source_checkpoint / 'config.json').is_file():
    raise FileNotFoundError(
        f'Selected {regression_source} checkpoint is missing config.json: '
        f'{source_checkpoint}'
    )
if not any(
    path.is_file()
    for pattern in ('*.safetensors', '*.bin', '*.safetensors.index.json', '*.bin.index.json')
    for path in source_checkpoint.glob(pattern)
):
    raise FileNotFoundError(
        f'Selected {regression_source} checkpoint has no model weights: '
        f'{source_checkpoint}'
    )

for filename in ['train.tsv', 'eval.tsv', 'test.tsv']:
    path = genex_dir / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty regression file: {path}')

print('Regression source:', regression_source)
print('Exact pretrained checkpoint consumed by regression:', source_checkpoint)
print('Regression output:', regression_output)

from transformers import AutoConfig, PreTrainedTokenizerFast

from module.florabert import config as flora_config
from module.florabert import transformers as flora_transformers
from module.florabert import utils as flora_utils


source_config = AutoConfig.from_pretrained(
    str(source_checkpoint),
    local_files_only=True,
)
regression_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(plant_tokenizer),
    local_files_only=True,
)

print('Checkpoint model type:', source_config.model_type)
print('Checkpoint vocab size:', source_config.vocab_size)
print('Tokenizer vocab size:', len(regression_tokenizer))
assert source_config.model_type == 'modernbert'
assert source_config.vocab_size == len(regression_tokenizer)

flora_config.reload_settings()
regression_settings = flora_utils.get_model_settings(
    flora_config.settings,
    model_name='modernbert-pred-mean-pool',
)
regression_settings['output_mode'] = 'regression'
regression_settings['num_labels'] = len(flora_config.tissues)

_, loaded_tokenizer, regression_model = flora_transformers.load_model(
    'modernbert-pred-mean-pool',
    str(plant_tokenizer),
    pretrained_model=str(source_checkpoint),
    log_offset=0.001,
    **regression_settings,
)

regression_params = flora_utils.count_model_parameters(
    regression_model,
    trainable_only=False,
)
print('Verified pretrained base weights loaded into the regression model.')
print('Regression model parameter count:', regression_params)

smoke_inputs = loaded_tokenizer(
    'ACGTACGTACGTTTTAAACCCGGG',
    return_tensors='pt',
    max_length=loaded_tokenizer.model_max_length,
    truncation=True,
    padding='max_length',
)
regression_model.eval()
with torch.no_grad():
    smoke_outputs = regression_model(**smoke_inputs)

assert smoke_outputs.logits.shape == (1, len(flora_config.tissues))
assert torch.isfinite(smoke_outputs.logits).all()
print('One regression forward pass:', tuple(smoke_outputs.logits.shape))

del regression_model, loaded_tokenizer, smoke_outputs, smoke_inputs

In [ ]:
finetune_settings = dict(flora_config.settings['training']['finetune'])
regression_learning_rate = (
    float(os.environ['FLORABERT_REGRESSION_LEARNING_RATE'])
    if os.environ.get('FLORABERT_REGRESSION_LEARNING_RATE')
    else None
)
regression_epochs = (
    int(os.environ['FLORABERT_REGRESSION_EPOCHS'])
    if os.environ.get('FLORABERT_REGRESSION_EPOCHS')
    else None
)
n_workers = int(os.environ.get('FLORABERT_DATA_WORKERS', '2'))
force_rerun = env_bool('FLORABERT_FORCE_RERUN', False)
regression_resume_from = (
    Path(os.environ['FLORABERT_REGRESSION_RESUME_FROM']).expanduser()
    if os.environ.get('FLORABERT_REGRESSION_RESUME_FROM')
    else None
)

print('Current repo regression settings:', finetune_settings)
print('Natural-log target: ln(TPM + 0.001)')
print(
    'Effective learning rate:',
    regression_learning_rate or finetune_settings['learning_rate'],
)
print(
    'Effective epochs:',
    regression_epochs or finetune_settings['num_train_epochs'],
)
print('Regression resume checkpoint:', regression_resume_from or 'none')
print('CUDA devices:', cuda_count)

In [ ]:
def launch_repo_script(relative_script, arguments):
    script_path = repo_dir / relative_script

    if cuda_count > 1:
        accelerate_exe = shutil.which('accelerate')
        if accelerate_exe:
            command = [
                accelerate_exe,
                'launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
        else:
            command = [
                sys.executable,
                '-m',
                'accelerate.commands.launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
    else:
        command = [sys.executable, '-u', str(script_path), *map(str, arguments)]

    environment = os.environ.copy()
    environment['PYTHONPATH'] = (
        str(repo_dir) + os.pathsep + environment.get('PYTHONPATH', '')
    )
    environment['PYTHONUNBUFFERED'] = '1'

    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(repo_dir),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def regression_arguments():
    arguments = [
        '--model-name',
        'modernbert-pred-mean-pool',
        '--data-dir',
        str(genex_dir),
        '--train-data',
        'train.tsv',
        '--eval-data',
        'eval.tsv',
        '--test-data',
        'test.tsv',
        '--tokenizer-dir',
        str(plant_tokenizer),
        '--output-dir',
        str(regression_output),
        '--transformation',
        'log',
        '--log-offset',
        '0.001',
        '--precision',
        'fp16' if cuda_count else 'no',
        '--n-workers',
        str(n_workers),
    ]

    if regression_resume_from is None:
        arguments.extend(['--pretrained-model', str(source_checkpoint)])
    else:
        if not regression_resume_from.is_dir():
            raise FileNotFoundError(
                f'Regression resume checkpoint does not exist: {regression_resume_from}'
            )
        if not (regression_resume_from / 'training_state.pt').is_file():
            raise FileNotFoundError(
                'Regression resume requires training_state.pt alongside the '
                f'checkpoint: {regression_resume_from}'
            )
        arguments.extend(['--resume-from-checkpoint', str(regression_resume_from)])

    if regression_learning_rate is not None:
        arguments.extend(
            ['--learning-rate', str(regression_learning_rate)]
        )
    if regression_epochs is not None:
        arguments.extend(['--num-train-epochs', str(regression_epochs)])

    return arguments

In [ ]:
best_config = regression_output / 'best' / 'config.json'
if (
    best_config.is_file()
    and not force_rerun
    and regression_resume_from is None
):
    print(
        'Regression output already exists; set FLORABERT_FORCE_RERUN=1 '
        'to retrain:',
        regression_output,
    )
else:
    launch_repo_script(
        Path('scripts/1-modeling/finetune.py'),
        regression_arguments(),
    )

print('Validation selection is performed after every epoch by finetune.py.')
print('The retained checkpoint is:', regression_output / 'best')

In [ ]:
import pandas as pd

metrics_file = regression_output / 'metrics.jsonl'
best_file = regression_output / 'best_metrics.json'
best_checkpoint = regression_output / 'best'

if not metrics_file.is_file():
    raise FileNotFoundError(f'Validation history is missing: {metrics_file}')
if not best_file.is_file() or not (best_checkpoint / 'config.json').is_file():
    raise RuntimeError(
        f'Best validation checkpoint is missing under {regression_output}'
    )

records = [
    json.loads(line)
    for line in metrics_file.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
history = pd.DataFrame(
    [
        {'epoch': record['epoch'], **record['overall']}
        for record in records
    ]
)

display(
    history[
        [
            'epoch',
            'mse',
            'r2',
            'pearson_r2',
            'prediction_mean',
            'prediction_std',
            'target_mean',
            'target_std',
        ]
    ]
)
print('Best-checkpoint metadata:')
print(best_file.read_text(encoding='utf-8'))
print('Best checkpoint:', best_checkpoint)

In [ ]:
import numpy as np
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

from module.florabert import dataio as flora_dataio


def population_std(values):
    return float(np.asarray(values, dtype='float64').std(ddof=0))


def metric_row(scope, targets, predictions):
    targets = np.asarray(targets, dtype='float64').reshape(-1)
    predictions = np.asarray(predictions, dtype='float64').reshape(-1)
    correlation = (
        float('nan')
        if targets.std() == 0 or predictions.std() == 0
        else float(np.corrcoef(targets, predictions)[0, 1])
    )

    return {
        'scope': scope,
        'mse': float(np.mean((targets - predictions) ** 2)),
        'sklearn_r2': float(r2_score(targets, predictions)),
        'pearson_r2': correlation ** 2 if correlation == correlation else float('nan'),
        'prediction_mean': float(predictions.mean()),
        'prediction_std': population_std(predictions),
        'target_mean': float(targets.mean()),
        'target_std': population_std(targets),
    }


def report_metrics(targets, predictions):
    rows = [metric_row('overall', targets, predictions)]

    for tissue_idx, tissue in enumerate(flora_config.tissues):
        rows.append(
            metric_row(
                tissue,
                targets[:, tissue_idx],
                predictions[:, tissue_idx],
            )
        )

    return pd.DataFrame(rows).set_index('scope')


if not (best_checkpoint / 'config.json').is_file():
    raise FileNotFoundError(f'Best checkpoint is missing: {best_checkpoint}')

best_settings = dict(regression_settings)
best_settings['output_mode'] = 'regression'
best_settings['num_labels'] = len(flora_config.tissues)
_, eval_tokenizer, eval_model = flora_transformers.load_model(
    'modernbert-pred-mean-pool',
    str(plant_tokenizer),
    pretrained_model=str(best_checkpoint),
    log_offset=0.001,
    **best_settings,
)

test_datasets = flora_dataio.load_datasets(
    eval_tokenizer,
    str(genex_dir / 'test.tsv'),
    seq_key='sequence',
    file_type='csv',
    delimiter='\t',
    transformation='log',
    log_offset=0.001,
    shuffle=False,
    n_workers=n_workers,
)
test_dataset = test_datasets['train'].remove_columns(['sequence'])
eval_batch_size = int(os.environ.get('FLORABERT_EVAL_BATCH_SIZE', '8'))
test_loader = DataLoader(
    test_dataset,
    batch_size=eval_batch_size,
    collate_fn=flora_dataio.load_data_collator('pred'),
    shuffle=False,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_model.to(device).eval()
predictions = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch['labels'].to(device)
        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key in {'input_ids', 'attention_mask', 'position_ids', 'labels'}
        }
        outputs = eval_model(**inputs)
        predictions.append(outputs.logits.detach().cpu())
        targets.append(labels.detach().cpu())

target_array = torch.cat(targets).numpy()
prediction_array = torch.cat(predictions).numpy()
results = report_metrics(target_array, prediction_array)
print(f'Test metrics for {regression_source} best checkpoint in log space:')
display(results)

evaluation_dir = run_root / 'evaluation'
evaluation_dir.mkdir(parents=True, exist_ok=True)
(evaluation_dir / f'{regression_source}_best.json').write_text(
    json.dumps(results.reset_index().to_dict(orient='records'), indent=2),
    encoding='utf-8',
)
print('Saved metrics:', evaluation_dir / f'{regression_source}_best.json')